In [2]:
print("\n🏗️  Society of Mind Architecture:")
print("""
    ┌─────────────────────────────────────-┐
    │        Society of Mind Agent         │
    │  ┌─────────────────────────────────┐ │
    │  │         Inner Team              │ │
    │  │  ┌─────────┐  ┌─────────────┐   │ │
    │  │  │ Agent A │  │   Agent B   │   │ │
    │  │  │(Writer) │  │  (Editor)   │   │ │
    │  │  └─────────┘  └─────────────┘   │ │
    │  │         │            │          │ │
    │  │         └────┬───────┘          │ │
    │  │              │                  │ │
    │  │         Discussion              │ │
    │  └─────────────────────────────────┘ │
    │                 │                    │
    │            Synthesis                 │
    │                 │                    │
    │           Final Response             │
    └─────────────────────────────────────-┘
    """)


🏗️  Society of Mind Architecture:

    ┌─────────────────────────────────────-┐
    │        Society of Mind Agent         │
    │  ┌─────────────────────────────────┐ │
    │  │         Inner Team              │ │
    │  │  ┌─────────┐  ┌─────────────┐   │ │
    │  │  │ Agent A │  │   Agent B   │   │ │
    │  │  │(Writer) │  │  (Editor)   │   │ │
    │  │  └─────────┘  └─────────────┘   │ │
    │  │         │            │          │ │
    │  │         └────┬───────┘          │ │
    │  │              │                  │ │
    │  │         Discussion              │ │
    │  └─────────────────────────────────┘ │
    │                 │                    │
    │            Synthesis                 │
    │                 │                    │
    │           Final Response             │
    └─────────────────────────────────────-┘
    


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
import asyncio
from autogen_agentchat.ui import Console
from autogen_agentchat.agents import AssistantAgent, SocietyOfMindAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination

In [5]:

async def main() -> None:
    model_client = OpenAIChatCompletionClient(
            model='gpt-4o-mini',
            temperature=0.0
    )

    agent1 = AssistantAgent("assistant1", model_client=model_client, system_message="You are a writer, write well.")
    agent2 = AssistantAgent(
        "assistant2",
        model_client=model_client,
        system_message="You are an editor, provide critical feedback. Respond with 'APPROVE' if the text addresses all feedbacks.",
    )
    
    inner_termination = TextMentionTermination("APPROVE")
    
    inner_team = RoundRobinGroupChat([agent1, agent2], termination_condition=inner_termination)

    society_of_mind_agent = SocietyOfMindAgent("society_of_mind", team=inner_team, model_client=model_client)

    agent3 = AssistantAgent(
        "assistant3", model_client=model_client, system_message="Translate the text to Spanish."
    )
    
    team = RoundRobinGroupChat([society_of_mind_agent, agent3], max_turns=2)

    stream = team.run_stream(task="Write a short story with a surprising ending. Give the story in just 50 words.")
    
    await Console(stream)
    

await (main())


---------- TextMessage (user) ----------
Write a short story with a surprising ending. Give the story in just 50 words.
---------- TextMessage (assistant1) ----------
In a quiet village, an old clock tower chimed every hour, its sound echoing through the streets. One night, a curious girl climbed to the top, discovering a hidden door. Inside, she found a room filled with clocks, all ticking backward. She realized time was never meant to be reversed.
---------- TextMessage (assistant2) ----------
APPROVE
---------- TextMessage (society_of_mind) ----------
In a quiet village, an old clock tower chimed every hour, its sound echoing through the streets. One night, a curious girl climbed to the top, discovering a hidden door. Inside, she found a room filled with clocks, all ticking backward. She realized time was never meant to be reversed.
---------- TextMessage (assistant3) ----------
En un tranquilo pueblo, una vieja torre del reloj sonaba cada hora, su eco resonando por las calles. Una 